# ETL Notebooks — Examples & Dataset Guide

A quick-reference guide to what was registered and why. Use this notebook to orient yourself before diving into a specific ETL notebook.

> **All notebooks live in `code/`.** Outputs land in `../scratch/em_patchseq_wnm_v1/`. Registry-backed model tables are written with `write_models(...)` (projection rows use `write_projection_matrix(...)`).

---
# Part 1 — Datasets & DataItems

## What is a DataItem?

A **DataItem** is one registered cell. Key properties:
- `id` — the original identifier from the source file, **never cast or modified**
- `project_id` — namespace used for partition scoping during writes
- Cells are linked to datasets via **`DataItemDataSetAssociation`** (many-to-many)

A cell can belong to multiple datasets (e.g. a nucleus that was also CSM-classified, or an inh patchseq cell that was also reconstructed morphologically).

---
## Minnie65 EM

`project_id = "minnie65"` &nbsp;|&nbsp; notebooks: `etl_minnie_01`, `etl_minnie_02`

### Two datasets, why?

| dataset_id | What it contains | Source |
|---|---|---|
| `minnie65_v1412_nuclei` | Every detected nucleus with `pt_root_id != 0` at materialization v1412 | CAVE `nucleus_detection_lookup_v1` |
| `minnie65_v1412_csm_cluster` | Subset that passed **CSM** (Clustering by Soma Morphology / dendrite ultrastructure) classification | `minnie_features.parquet` |

- **Nuclei** = ground truth for cell existence. If a nucleus is detected, it gets a DataItem.
- **CSM subset** = an analysis result. Not every cell was reconstructed well enough for CSM; only classified cells enter this dataset. Features and cluster labels are only meaningful for this subset.
- The two datasets share the same DataItems for cells that appear in both — the same `id` value, joined via association rows.

---
## VISp Patch-seq — inhibitory and excitatory

`project_id = "visp_patchseq"` (shared) &nbsp;|&nbsp; notebooks: `etl_visp_inh_patchseq_01`, `etl_visp_exc_patchseq_01`, and their `_02` counterparts

Both inhibitory and excitatory cells share one `project_id` but have **separate `dataset_id`s**.

| dataset_id | Source | Cells registered | Why this many |
|---|---|---|---|
| `visp_inh_patchseq` | `patchseq_tx_cell_ttype_labels.csv` | 2,759 | All cells that received a transcriptomic **T-type label** — a broad criterion |
| `visp_exc_patchseq` | `inferred_met_types.csv` | 1,528 | Cells that passed the full **MET-type inference** pipeline (multimodal: electrophysiology + morphology + transcriptomics) — a stricter filter |


### The 120 extra inhibitory cells

The morphology feature file (`inh_ivscc_features_wide_unnormalized.csv`) contained **520 cells** with reconstructed morphologies. Of those, 400 were already in `visp_inh_patchseq` (registered by `_01`), but **120 were not**.

These 120 cells had successful morphology reconstructions but didn't appear in the T-type label source — likely borderline QC or later additions to the reconstruction pipeline. They are registered (DataItem + association) when `etl_visp_inh_patchseq_02` runs, not `_01`.

> All 389 excitatory morphology cells were already in `_01` — no extras.

---
## WNM Excitatory (Whole-Neuron Morphology)

`project_id = "visp_wnm"` &nbsp;|&nbsp; notebooks: `etl_wnm_exc_01`, `etl_wnm_exc_02`

| dataset_id | Source | Cells registered |
|---|---|---|
| `visp_exc_wnm` | `FullMorphMetaData_Master.csv` | 341 (by `_01`) + 4 extras (by `_02`) = **345 total** |

- The 4 extra cells were found in the feature CSVs (`RawFeaturesWide_ChamferCorr.csv` / `AxonRawReatureWide.csv`) but not in the master metadata file. Same late-addition pattern as the 120 extra inh cells.

---
## Dataset summary

| dataset_id | project_id | Cells from `_01` | Extras added by `_02` | Modality |
|---|---|---|---|---|
| `minnie65_v1412_nuclei` | `minnie65` | all CAVE nuclei | — | Electron microscopy |
| `minnie65_v1412_csm_cluster` | `minnie65` | CSM-classified subset | — | EM |
| `visp_inh_patchseq` | `visp_patchseq` | 2,759 | +120 (`_02`), +103 (`_03`) | Morphology (patch-seq) |
| `visp_exc_patchseq` | `visp_patchseq` | 1,528 | 0 | Morphology (patch-seq) |
| `visp_exc_wnm` | `visp_wnm` | 341 | +4 | Morphology (whole-neuron) |

> The 103 extra inhibitory cells come from the MET-type assignment file (`visp_met_cell_assignments_text_names.csv`) read by `etl_visp_inh_patchseq_03`. Same late-addition pattern as the 120 from `_02` and the 4 from WNM `_02`/`_04`.


---
## Reference taxonomies (no DataItems)

Two `_01` notebooks register **global cluster taxonomies** without registering any cells. They are reference artefacts — consumed by `_03` notebooks that assign project cells to these taxonomies.

| Notebook | `hierarchy_id` | What it owns |
|---|---|---|
| `etl_tasic_01_cluster.ipynb` | `tasic_2018_visp_taxonomy` | Tasic 2018 VISp scRNA-seq taxonomy (class → subclass → cluster) |
| `etl_visp_met_types_01_cluster.ipynb` | `visp_met_types_taxonomy` | VISp MET-types (class → cluster), 45 leaves |

Both write `algorithmrun/`, `clusterhierarchy/`, `cluster/`, and `hierarchycategory/` rows. No `project_id`; `Cluster` rows are scoped by `hierarchy_id`, while the others are id-scoped in the write registry.


---
# Part 2 — Feature Sets

## Overview

| feature_set_id | project_id(s) | Cells | Features | Owning notebook |
|---|---|---|---|---|
| `inh_visp_morph_features` | `visp_patchseq` | 520 | 46 | `etl_visp_inh_patchseq_02` |
| `exc_visp_morph_features` | `visp_patchseq` **+** `visp_wnm` | 389 + 345 | 50 | defs/set: `etl_visp_exc_patchseq_02`; WNM rows: `etl_wnm_exc_02` |
| `wnm_exc_local_axon_features` | `visp_wnm` | 345 | 51 | `etl_wnm_exc_02` |
| `wnm_exc_complete_axon_features` | `visp_wnm` | 341 | 18 | `etl_wnm_exc_02` |
| `csm_cluster_features` | `minnie65` | CSM subset | 82 | `etl_minnie_02` |
| `minnie65_std_transform_coordinates` | `minnie65` | all nuclei | 3 (x, y, z µm) | `etl_minnie_02` |

---
## The shared `exc_visp_morph_features` set

This is the most nuanced ownership pattern in the project.

- **Definitions and `CellFeatureSet`** are owned by `etl_visp_exc_patchseq_02` (`project_id="visp_patchseq"`). They are written once, not duplicated.
- **`etl_wnm_exc_02`** reads those definitions back from `cellfeaturedefinition/`, then writes its own 345 rows into `cellfeatures/exc_visp_morph_features/` with `project_id="visp_wnm"`.
- The resulting Delta table has **two blocks of rows** sharing the same `feature_set_id` but different `project_id`s.

```
cellfeatures/exc_visp_morph_features/
  project_id=visp_patchseq/   ← 389 exc patchseq cells
  project_id=visp_wnm/        ← 345 WNM cells
```

**Why?** The two populations share the same morphological feature definitions (skeleton_keys features) and should be comparable. Keeping them in one Delta table makes cross-population analysis straightforward.

**Caveat:** 6 of the 50 features are absent from the WNM CSV — those columns are `NaN` for all WNM rows. A warning is printed when `etl_wnm_exc_02` runs.

> To query only one project: `pl.read_delta("cellfeatures/exc_visp_morph_features/").filter(pl.col("project_id") == "visp_wnm")`

---
## WNM-only feature sets

WNM has three feature sets in total:

| Feature set | What it captures | Definition source |
|---|---|---|
| `exc_visp_morph_features` | Shared apical/basal dendrite morphology features | CSV (`exc_visp_patchseq_morph_feature_definitions.csv`), owned by exc patchseq `_02` |
| `wnm_exc_local_axon_features` | Local axon + apical dendrite features specific to WNM | Built from column names of `AxonRawReatureWide.csv` (`data_type="<f8"`) |
| `wnm_exc_complete_axon_features` | Whole-brain axon features from fMOST reconstructions | Built from column names of `fMOST_Complete_Axon_Features.csv` (`data_type="<f8"`) |

For `local_axon` and `complete_axon`: there was no separate definition CSV, so `CellFeatureDefinition` rows are generated directly from the wide-CSV column names with `data_type="<f8"` and blank `unit`/`description`. These can be enriched manually later.

---
# Part 3 — Cluster Taxonomies & Memberships

## What's a taxonomy?

A named hierarchy of cluster labels (root → classes → leaves) plus the algorithm run that produced it. Stored globally and identified by `hierarchy_id`. Three taxonomies are registered:

- `tasic_2018_visp_taxonomy` — Tasic 2018 VISp scRNA-seq taxonomy.
- `visp_met_types_taxonomy` — VISp MET-types (multimodal Patch-seq).
- `minnie65_csm_cell_types` — Minnie65 CSM cell-type taxonomy.


## Membership vs mapping

Same data shape (cell → cluster), different meaning. Downstream queries should distinguish:

- **Membership** (`ClusterMembership`) — the cell *belongs to* this cluster by definition. Used when the cell was part of the cohort that **defined** the taxonomy.
- **Mapping** (`CellToClusterMapping` + a `MappingSet` row that names the method) — the cell was *assigned* to this cluster by some classifier after the fact. Used when the cell was **not** part of the original cohort.

Example: VISp MET-types were defined by inhibitory and excitatory Patch-seq cells, so those cells get **memberships**. WNM cells were classified into MET-types later by a random forest, so they get **mappings**.


## Parent propagation

Every `_03` notebook writes one row per (cell × ancestor) all the way up to the root. Queries like *"which cells are class L4 IT?"* don't need to join the hierarchy table.

`probability` is set on the leaf row only and left null on parents — the leaf is what the algorithm decided; ancestors are bookkeeping.


## Per-project assignments

| Project | Cohort | Hierarchy | Notebook | Type |
|---|---|---|---|---|
| `visp_patchseq` | exc | `tasic_2018_visp_taxonomy` | `etl_visp_exc_patchseq_03` | mapping (`t_type`, with `ET → PT` rename) |
| `visp_patchseq` | exc | `visp_met_types_taxonomy`  | `etl_visp_exc_patchseq_03` | membership (`met_type`) |
| `visp_patchseq` | inh | `tasic_2018_visp_taxonomy` | `etl_visp_inh_patchseq_03` | mapping (`ttype`) |
| `visp_patchseq` | inh | `visp_met_types_taxonomy`  | `etl_visp_inh_patchseq_03` | membership (`met_type`) |
| `visp_wnm`      | exc | `visp_met_types_taxonomy`  | `etl_wnm_exc_03`           | mapping (RF classifier, `probability` per call) |
| `minnie65`      | CSM | `minnie65_csm_cell_types`  | `etl_minnie_03`            | membership (35,780 cells) |

Both Patch-seq `_03`s write the **same** `(visp_patchseq, visp_met_types_taxonomy)` slice of `clustermembership/`. They merge their cohorts on every run, so order doesn't matter and re-running either is safe.

**Minnie's taxonomy is locally owned** — `etl_minnie_03` registers both the taxonomy *and* the memberships in the same notebook, because no separate cohort exists to own the CSM taxonomy on its own. The other taxonomies are owned by their `_01`s.


---
# Part 4 — Connectivity & Projections

Two `_04` notebooks add cell-pair and per-cell measurement tables.


## `etl_minnie_04_cell_cell.ipynb`

`CellCellConnectivityLong` for Minnie65 v1412. Source: precomputed soma-soma connectivity parquet. The notebook also builds a proofread cohort by querying CAVE `proofreading_status_and_strategy` and intersecting with the CSM cohort.

Two example output folders (one notebook, two filters):

- `cellcellconnectivitylong_proofread_pre_to_csm_post/` — proofread cells as senders, CSM cells as receivers. Records both `SYNAPSE_COUNT` and `SUM_ANATOMICAL_SIZE`.
- `cellcellconnectivitylong_proofread_to_proofread/` — only proofread cells on both sides. `SYNAPSE_COUNT` only.

> Two folders, not one, because both examples share `project_id` and would overwrite each other if written into a single Delta table. A future schema addition (e.g. `connectome_id`) could merge them.


## `etl_wnm_exc_04_projection_matrix.ipynb`

Two `ProjectionMeasurementMatrix` rows for WNM excitatory cells, plus the underlying wide-parquet tables:

- `wnm_exc_proj_ipsi`   — 345 cells × 152 ipsilateral region acronyms.
- `wnm_exc_proj_contra` — 345 cells × 68 contralateral region acronyms.

Source: `ProjectionMatrix_tip_and_branch_roll_up.csv`. Cell ids are the SWC filename with `.swc` stripped (matches `_01`).

Adds **+4 new cells** found in the projection CSV but not yet in `dataitem/` — the same late-addition pattern as the `_02` notebooks. Registered via `write_models(DataItem(...))` (append-new-by-id mode).
